# 🚀 KTRS 마케팅 봇 전용 구글 무료 GPU (RealVisXL + 동일 인물 락 무결점 안정 버전)
이 노트북은 **Google Colab 무료 GPU(Tesla T4)**를 활용하여 **RAM 7.5GB 초경량 0원 무제한 극실사 숏폼**을 생성하는 서버입니다.

### 📌 사용 방법:
1. 상단 메뉴 **[런타임 ➔ 모두 실행 (Ctrl+F9)]** 클릭
2. 약 1분 후 아래에 출력되는 `공용 URL`을 복사하면 끝!

In [ ]:
# 1. 필수 라이브러리 및 Cloudflare 터널 자동 설치
!pip install -q diffusers transformers accelerate safetensors fastapi uvicorn pillow pydantic torchvision
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 2. RealVisXL V4.0 + 동일 인물 락(img2img) 모델 로딩 & 서버 가동
import io
import os
import re
import time
import base64
import random
import threading
import subprocess
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
from PIL import Image
import uvicorn
from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, DPMSolverMultistepScheduler

app = FastAPI(title='KTRS RealVisXL GPU Image Server')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 가동 GPU: {torch.cuda.get_device_name(0) if device == "cuda" else "CPU"}')

# 1) RealVisXL V4.0 베이스 로딩 (가벼운 7.5GB, 튕김 0%)
MODEL_ID = 'SG161222/RealVisXL_V4.0'
pipe_text = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    variant='fp16' if device == 'cuda' else None
)
pipe_text.scheduler = DPMSolverMultistepScheduler.from_config(pipe_text.scheduler.config, use_karras_sigmas=True)
pipe_text = pipe_text.to(device)
if device == 'cuda':
    pipe_text.enable_attention_slicing()

# 2) 동일 인물 락 img2img 파이프라인 (추가 메모리 0MB, 가중치 100% 공유)
pipe_img2img = AutoPipelineForImage2Image.from_pipe(pipe_text)
print('✅ RealVisXL 극실사 AI 모델 + 동일 인물 락(Zero-RAM) 준비 완료!')

class ImageGenRequest(BaseModel):
    prompt: str
    negative_prompt: Optional[str] = 'caucasian, white, deformed fingers, extra limbs, bad anatomy, ugly, blurry, 3d render, cartoon, plastic skin'
    aspect_ratio: Optional[str] = '9:16'
    seed: Optional[int] = -1
    guidance_scale: Optional[float] = 5.0
    num_inference_steps: Optional[int] = 25
    ref_image_base64: Optional[str] = None

@app.get('/')
def health():
    return {'status': 'ok', 'engine': 'RealVisXL V4.0 + Face Lock', 'device': device}

@app.post('/generate')
def generate_image(req: ImageGenRequest):
    try:
        if req.aspect_ratio == '9:16':
            width, height = 768, 1344
        elif req.aspect_ratio == '1:1':
            width, height = 1024, 1024
        elif req.aspect_ratio == '16:9':
            width, height = 1344, 768
        else:
            width, height = 768, 1344

        used_seed = req.seed if req.seed is not None and req.seed >= 0 else random.randint(100000, 999999999)
        generator = torch.Generator(device=device).manual_seed(used_seed)

        # 🔒 동일 인물 락 검사 (씬 1 사진이 있으면 씬 1 얼굴 뼈대 기반 렌더링)
        ref_image = None
        if req.ref_image_base64:
            try:
                ref_bytes = base64.b64decode(req.ref_image_base64)
                ref_image = Image.open(io.BytesIO(ref_bytes)).convert('RGB')
                ref_image = ref_image.resize((width, height), Image.LANCZOS)
                print('🔒 [얼굴 락 발동] 씬 1 인물 뼈대 기반 렌더링!')
            except Exception as e:
                print(f'참조 이미지 디코딩 실패: {e}')
                ref_image = None

        if ref_image is not None:
            image = pipe_img2img(
                prompt=req.prompt,
                negative_prompt=req.negative_prompt,
                image=ref_image,
                strength=0.55,
                guidance_scale=req.guidance_scale,
                num_inference_steps=req.num_inference_steps,
                generator=generator
            ).images[0]
        else:
            image = pipe_text(
                prompt=req.prompt,
                negative_prompt=req.negative_prompt,
                width=width,
                height=height,
                guidance_scale=req.guidance_scale,
                num_inference_steps=req.num_inference_steps,
                generator=generator
            ).images[0]

        buffered = io.BytesIO()
        image.save(buffered, format='JPEG', quality=95)
        img_str = base64.b64encode(buffered.getvalue()).decode('utf-8')

        return {'success': True, 'seed': used_seed, 'image_base64': img_str, 'width': width, 'height': height, 'face_locked': bool(ref_image is not None)}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 백그라운드 스레드로 FastAPI 서버 가동
def start_uvicorn():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)

# Cloudflare 무설정 터널 가동
if not os.path.exists('cloudflared'):
    os.system('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared')

proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)

public_url = ''
for line in iter(proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print(f'\n========================================================')
print(f'🎉 [구글 코랩 무료 GPU(RealVisXL + 동일 인물 락) 가동 완료!]')
print(f'👉 공용 URL: {public_url}')
print(f'========================================================\n')
